# CAREPILOT — Action Recommendation Benchmark

*Chapter 2 — État de l'art et benchmarking IA · CAREPILOT operational AI*

## 1. Objective

Recommend the **next operational action** (`recommended_action`) an N1/N2 agent should take on a ticket.

## 2. Problem Type


- **Multiclass classification** (6 operational action classes)
- Classes: RESOLVE_LOCALLY, APPLY_KM, ESCALATE_N3, TRANSFER_PERIMETER, REQUEST_INFORMATION, MONITOR
- Primary metric: **Macro F1**

## 3. Dataset Description


- **Source:** `ml_training.historical_tickets` (14,000 tickets)
- **Class distribution:** RESOLVE_LOCALLY 21.8%, APPLY_KM 20.6%, TRANSFER_PERIMETER 17.3%, ESCALATE_N3 16.7%, REQUEST_INFORMATION 13.0%, MONITOR 10.6%
- **Split:** 9800 train / 2100 validation / 2100 test (stratified)

## 4. Preprocessing

Title + description TF-IDF; one-hot categoricals; numeric imputation; label encoding for six action classes.

## 5. Models / Methods Compared

Logistic Regression · Random Forest · XGBoost

This benchmark compares multiple approaches in order to select the most suitable method for integration into CAREPILOT.

## 6. Evaluation Metrics

Accuracy · Macro/Weighted Precision · Macro/Weighted Recall · Macro F1 · Weighted F1 · Training/inference time

## 7. Training or Retrieval Pipeline

Multiclass sklearn pipelines with Macro F1 selection on held-out test split.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, Markdown, display

NOTEBOOK_DIR = Path.cwd()
ROOT = NOTEBOOK_DIR
for parent in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
    if (parent / "ml" / "artifacts").exists():
        ROOT = parent
        break

SHARED = ROOT / "ml" / "notebooks" / "_shared"
if str(SHARED) not in sys.path:
    sys.path.insert(0, str(SHARED))

from benchmark_notebook_helpers import display_plot, find_project_root, plot_results_table

ROOT = find_project_root()
ARTIFACTS = ROOT / "ml" / "artifacts"
PLOTS = ROOT / "ml" / "plots"
REPORTS = ROOT / "ml" / "reports"
MODELS = ARTIFACTS / "models"

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

RUN_BENCHMARK = False  # Set True to re-train / re-run the full benchmark pipeline


In [ ]:
BENCHMARK_SCRIPT = ROOT / "ml/benchmarks/action_recommendation/benchmark.py"
if RUN_BENCHMARK:
    cmd = [sys.executable, str(BENCHMARK_SCRIPT), "--project-root", str(ROOT), *[]]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("Skipping benchmark run (RUN_BENCHMARK=False). Loading saved artifacts.")
    print("Re-run with RUN_BENCHMARK=True or execute:")
    print(f"  python {BENCHMARK_SCRIPT} --project-root {ROOT}")


## 8. Benchmark Execution — Load Results

In [ ]:
metrics_path = ARTIFACTS / "03_action_recommendation_benchmark_metrics.json"
if not metrics_path.exists():
    raise FileNotFoundError(f"Missing metrics artifact: {metrics_path}")
payload = json.loads(metrics_path.read_text(encoding="utf-8"))
rows = payload.get("results", [])
results_df = pd.DataFrame(rows)
display(results_df)


## 9. Benchmark Results

Final comparison table (report-ready PNG + DataFrame):

In [ ]:
table_png = PLOTS / "03_action_recommendation_benchmark_results_table.png"
display_plot(table_png)
display_plot(PLOTS / "03_action_distribution.png")
display_plot(PLOTS / "03_action_model_comparison.png")
display_plot(PLOTS / "03_action_macro_f1.png")


## 10. Visualizations

Key plots are loaded from `ml/plots/` (generated by the benchmark pipeline). Re-run with `RUN_BENCHMARK=True` to regenerate.

## 11. Interpretation


XGBoost achieves the highest Macro F1 (~0.66), balancing performance across all six action classes. Logistic Regression remains interpretable but underperforms on minority classes. Random Forest sits between both.

Macro F1 ensures rare actions (MONITOR, REQUEST_INFORMATION) contribute equally to model selection — critical for enterprise action routing fairness.

## 12. Selected Model / Method

**Selected for CAREPILOT:**

**XGBoost** — best Macro F1. Artifact: `ml/artifacts/models/xgboost_action.joblib`.

## 13. Limitations

- Action labels reflect historical operator decisions, not always optimal
- Class imbalance affects minority action recall
- Combined routing depends on Tasks 1–2 predictions in production

## 14. Generated Artifacts

- `ml/artifacts/03_action_recommendation_benchmark_metrics.json`
- `ml/plots/03_action_recommendation_benchmark_results_table.png`
- `ml/reports/03_action_recommendation_benchmark_summary.md`